---
title: "广度优先搜索 (BFS)——多源扩散"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [1]:
#| code-fold: true
from collections import deque
from typing import List

## 📝 题目 994：腐烂的橘子

::: {.callout-caution}
### 题目要求
**描述**：在给定的 `m x n` 网格 `grid` 中，每个单元格可以有以下三个值之一：
- `0` 代表空单元格；
- `1` 代表新鲜橘子；
- `2` 代表腐烂的橘子。
每分钟，腐烂的橘子周围 **4 个正方向**（上下左右）相邻的新鲜橘子都会腐烂。
返回直到单元格中没有新鲜橘子为止所必须经过的 **最小分钟数**。如果不可能，返回 `-1`。

**输入输出示例**：

* **示例 1**：
    * **输入**：`grid = [[2,1,1],[1,1,0],[0,1,1]]`
    * **输出**：`4`

* **示例 2**：
    * **输入**：`grid = [[2,1,1],[0,1,1],[1,0,1]]`
    * **输出**：`-1`

* **示例 3**：
    * **输入**：`grid = [[0,2]]`
    * **输出**：`0`
:::

In [2]:
class Solution994:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        rows, cols = len(grid), len(grid[0])
        queue = deque()
        minute = 0
        fresh_count = 0
        for r in range(rows):  # 扫描矩阵，初始化多源起点和新鲜橘子总数
            for c in range(cols):
                if grid[r][c] == 2:
                    queue.append((r, c))
                elif grid[r][c] == 1:
                    fresh_count += 1
        if fresh_count == 0:  # 如果一开始就没有新鲜橘子，直接返回 0
            return 0
        while queue and fresh_count > 0:
            minute += 1
            for _ in range(len(queue)):
                r, c = queue.popleft()
                for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:  # 边界检查与新鲜判定
                        grid[nr][nc] = 2  # 感染
                        fresh_count -= 1  # 新鲜橘子减少
                        queue.append((nr, nc))
        return minute if fresh_count == 0 else -1

In [11]:
#| code-fold: true
def test_994(sol_func):
    test_cases = [
        {"grid": [[2,1,1],[1,1,0],[0,1,1]], "expected": 4, "desc": "标准连通扩散"},
        {"grid": [[2,1,1],[0,1,1],[1,0,1]], "expected": -1, "desc": "存在无法被感染的孤岛"},
        {"grid": [[0,2]], "expected": 0, "desc": "初始即无新鲜橘子"},
        {"grid": [[1]], "expected": -1, "desc": "全是好橘子但无传染源"},
        {"grid": [[2,1,1],[1,1,1],[0,1,2]], "expected": 2, "desc": "两端同时向中间扩散"},
        {"grid": [[2,1,0,1]], "expected": -1, "desc": "单行中被空格阻断"},
        {"grid": [[2,1],[1,1]], "expected": 2, "desc": "2x2 矩阵全满"},
        {"grid": [[2]], "expected": 0, "desc": "只有一个烂橘子"},
        {"grid": [[0]], "expected": 0, "desc": "只有一个空格"},
        {"grid": [[0,0],[0,0]], "expected": 0, "desc": "空旷地带"}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 70)
    for i, tc in enumerate(test_cases):
        input_grid = [row[:] for row in tc["grid"]]
        actual = sol_func(input_grid)
        status = "✅ PASS" if actual == tc["expected"] else "❌ FAIL"
        if actual == tc["expected"]: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")
    print("-" * 70)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_994(Solution994().orangesRotting)

ID   | 结果     | 预期   | 实际   | 描述
----------------------------------------------------------------------
1    | ✅ PASS | 4    | 4    | 标准连通扩散
2    | ✅ PASS | -1   | -1   | 存在无法被感染的孤岛
3    | ✅ PASS | 0    | 0    | 初始即无新鲜橘子
4    | ✅ PASS | -1   | -1   | 全是好橘子但无传染源
5    | ✅ PASS | 2    | 2    | 两端同时向中间扩散
6    | ✅ PASS | -1   | -1   | 单行中被空格阻断
7    | ✅ PASS | 2    | 2    | 2x2 矩阵全满
8    | ✅ PASS | 0    | 0    | 只有一个烂橘子
9    | ✅ PASS | 0    | 0    | 只有一个空格
10   | ✅ PASS | 0    | 0    | 空旷地带
----------------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

1. **初始化（寻找所有源头）**：
   - 遍历整个 $m \times n$ 矩阵。
   - **多源入队**：将所有初始状态为“腐烂”（值为 `2`）的橘子坐标 `(r, c)` 全部放入队列 `q`。
   - **计数新鲜橘子**：同时统计矩阵中“新鲜橘子”（值为 `1`）的总数 `fresh_count`。
   - 特殊情况：若 `fresh_count == 0`，说明没有新鲜橘子需要被感染，直接返回 `0`。

2. **多源层序遍历（按分钟扩散）**：
   - 初始化时间变量 `minutes = 0`。
   - **循环条件**：只要队列不为空 **且** 还有新鲜橘子（`fresh_count > 0`），就继续搜索。
     - *注意*：若没有新鲜橘子了，即便队列里还有腐烂橘子，也不应再增加分钟数。
   - **层级锁定**：使用 `for _ in range(len(q))`，确保这一分钟内，所有已腐烂的橘子同时向四周扩散。
   - **扩散逻辑**：
     - 弹出当前腐烂橘子的坐标，检查其 **上下左右** 4 个方向的邻居。
     - 如果邻居是新鲜橘子（值为 `1`）：
       - 将该位置的值改为 `2`（标记为腐烂）。
       - `fresh_count` 减 1。
       - 将新腐烂的坐标入队，以便下一分钟继续扩散。
   - **步数递增**：处理完当前层所有节点后，`minutes` 增加 1。

3. **最终判定**：
   - 当搜索结束，检查 `fresh_count`。
   - 若 `fresh_count == 0`，说明所有橘子都腐烂了，返回 `minutes`。
   - 若 `fresh_count > 0`，说明有些新鲜橘子被空格 `0` 孤立，无法被感染，返回 `-1`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(m \times n)$。每个单元格最多入队和出队一次。
* **空间复杂度**：$O(m \times n)$。在最坏情况下（全是腐烂橘子），队列中需要存储所有坐标。
:::

## 📝 题目 542：01 矩阵

::: {.callout-caution}
### 题目要求
**描述**：给定一个由 `0` 和 `1` 组成的矩阵 `mat` ，请输出一个大小相同的矩阵，其中每一个单元格的值都是原矩阵中该位置到 **最近的 0** 的距离。两个相邻单元格的距离为 `1` 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`mat = [[0,0,0],[0,1,0],[0,0,0]]`
    * **输出**：`[[0,0,0],[0,1,0],[0,0,0]]`

* **示例 2**：
    * **输入**：`mat = [[0,0,0],[0,1,0],[1,1,1]]`
    * **输出**：`[[0,0,0],[0,1,0],[1,2,1]]`

* **示例 3**：
    * **输入**：`mat = [[0,1,1],[1,1,1],[1,1,1]]`
    * **输出**：`[[0,1,2],[1,2,3],[2,3,4]]`
:::

In [5]:
class Solution542:
    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        rows, cols = len(mat), len(mat[0])
        dist = [[-1 for _ in range(cols)] for _ in range(rows)]  # 使用 -1 标记未访问过的点
        queue = deque()
        for r in range(rows):  # 多源起点初始化：将所有 0 入队，距离设为 0
            for c in range(cols):
                if mat[r][c] == 0:
                    dist[r][c] = 0
                    queue.append((r, c))
        while queue:
            r, c = queue.popleft()
            for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and dist[nr][nc] == -1:  # 边界检查 + 是否未访问检查
                    dist[nr][nc] = dist[r][c] + 1  # 关键逻辑：邻居的距离就是当前点的距离 + 1
                    queue.append((nr, nc))
        return dist

In [10]:
#| code-fold: true
def test_542(func):
    test_cases = [
        {"mat": [[0,0,0],[0,1,0],[0,0,0]], "expected": [[0,0,0],[0,1,0],[0,0,0]], "desc": "全包围：1 在中间"},
        {"mat": [[0,0,0],[0,1,0],[1,1,1]], "expected": [[0,0,0],[0,1,0],[1,2,1]], "desc": "非平衡扩散：下方的 1 距离较远"},
        {"mat": [[0,1,1],[1,1,1],[1,1,1]], "expected": [[0,1,2],[1,2,3],[2,3,4]], "desc": "单起点扩散：从左上角 0 向右下角扩散"},
        {"mat": [[0,0,0],[0,0,0]], "expected": [[0,0,0],[0,0,0]], "desc": "全 0 矩阵"},
        {"mat": [[0,1,1,0,1]], "expected": [[0,1,1,0,1]], "desc": "单行矩阵"},
        {"mat": [[1],[1],[0],[1]], "expected": [[2],[1],[0],[1]], "desc": "单列矩阵"},
        {"mat": [[0,1,0],[1,0,1],[0,1,0]], "expected": [[0,1,0],[1,0,1],[0,1,0]], "desc": "棋盘格（密集 0）"},
        {"mat": [[0,1,1],[1,1,1],[1,1,0]], "expected": [[0,1,2],[1,2,1],[2,1,0]], "desc": "蛇形路径（强迫波纹绕路）"},
        {"mat": [[0,1,1,1,0]], "expected": [[0,1,2,1,0]], "desc": "所有的 1 都在中间，0 在两头"},
        {"mat": [[1,1,1],[1,0,1],[1,1,1]], "expected": [[2,1,2],[1,0,1],[2,1,2]], "desc": "大规模全 1 中间夹一个 0"}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 40)
    for i, tc in enumerate(test_cases):
        mat_input = [row[:] for row in tc["mat"]]
        actual = func(mat_input)
        status = "✅ PASS" if actual == tc["expected"] else "❌ FAIL"
        if actual == tc["expected"]: passed_count = 0
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")
        if actual != tc["expected"]:
            print(f"   预期: {tc['expected']}")
            print(f"   实际: {actual}")
        else:
            passed += 1
    print("-" * 40)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_542(Solution542().updateMatrix)

ID   | 结果     | 描述
----------------------------------------
1    | ✅ PASS | 全包围：1 在中间
2    | ✅ PASS | 非平衡扩散：下方的 1 距离较远
3    | ✅ PASS | 单起点扩散：从左上角 0 向右下角扩散
4    | ✅ PASS | 全 0 矩阵
5    | ✅ PASS | 单行矩阵
6    | ✅ PASS | 单列矩阵
7    | ✅ PASS | 棋盘格（密集 0）
8    | ✅ PASS | 蛇形路径（强迫波纹绕路）
9    | ✅ PASS | 所有的 1 都在中间，0 在两头
10   | ✅ PASS | 大规模全 1 中间夹一个 0
----------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

1. **逆向思维与初始化**：
   - **问题转换**：从每个 `1` 开始找 `0` 效率极低。由于所有 `0` 到自身的距离都是 `0`，我们可以将所有 `0` 看作 **BFS 的多个起点**，同时向外层层扩散。
   - **预处理结果矩阵**：创建一个与原矩阵同大小的 `dist` 矩阵。
     - 若 `mat[i][j] == 0`，则 `dist[i][j] = 0`，并将该坐标 `(i, j)` 入队。
     - 若 `mat[i][j] == 1`，则 `dist[i][j] = -1`（表示该点尚未被“0 的波纹”触碰到）。

2. **多源 BFS 扩散**：
   - 只要队列不为空，弹出当前坐标 `(r, c)`。
   - 遍历 `(r, c)` 的 **上下左右** 四个邻居 `(nr, nc)`。
   - **判定条件**：如果邻居在矩阵范围内，且 `dist[nr][nc] == -1`：
     - 说明这是该 `1` 第一次被某个 `0` 触碰到。根据 BFS 的特性，这一定是 **最短距离**。
     - 更新 `dist[nr][nc] = dist[r][c] + 1`。
     - 将邻居坐标 `(nr, nc)` 入队，继续向更远的地方扩散。

3. **结果返回**：
   - 当队列为空时，所有的 `1` 都已被访问过，`dist` 矩阵即为最终结果。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(M \times N)$。矩阵中每个单元格仅入队和出队一次。
* **空间复杂度**：$O(M \times N)$。主要用于存储队列和结果矩阵 `dist`。
:::

## 📝 题目 1162：地图分析

::: {.callout-caution}
### 题目要求
**描述**：在 $n \times n$ 的网格 `grid` 中，`1` 代表陆地，`0` 代表海洋。请你找出一个海洋单元格，使得它离最近的陆地单元格的距离最大，并返回该距离。如果网格上只有陆地或者只有海洋，请返回 `-1`。
注：这里的距离是曼哈顿距离：$|x1 - x2| + |y1 - y2|$。

**输入输出示例**：

* **示例 1**：
    * **输入**：`grid = [[1,0,1],[0,0,0],[1,0,1]]`
    * **输出**：`2` （解释：海洋单元格 (1,1) 到最近陆地距离为 2）

* **示例 2**：
    * **输入**：`grid = [[1,0,0],[0,0,0],[0,0,0]]`
    * **输出**：`4` （解释：海洋单元格 (2,2) 到最近陆地距离为 4）

* **示例 3**：
    * **输入**：`grid = [[1,1,1],[1,1,1]]`
    * **输出**：`-1`
:::

In [8]:
class Solution1162:
    def maxDistance(self, grid: List[List[int]]) -> int:
        n = len(grid)
        queue = deque()
        dist = [[-1 for _ in range(n)] for _ in range(n)]
        result = -1
        for r in range(n):  # 初始化：将所有陆地入队
            for c in range(n):
                if grid[r][c] == 1:
                    dist[r][c] = 0
                    queue.append((r, c))
        if len(queue) == 0 or len(queue) == n * n:  # 边界检查：全陆地或全海洋返回 -1
            return -1
        while queue:
            r, c = queue.popleft()
            for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nr, nc = r + dr, c + dc
                if 0 <= nc < n and 0 <= nr < n and dist[nr][nc] == -1:
                    dist[nr][nc] = dist[r][c] + 1
                    queue.append((nr, nc))
                    result = max(result, dist[nr][nc])
        return result

In [14]:
#| code-fold: true
def test_1162(func):
    test_cases = [
        {"grid": [[1,0,1],[0,0,0],[1,0,1]], "expected": 2, "desc": "标准多源：中间海洋最远"},
        {"grid": [[1,0,0],[0,0,0],[0,0,0]], "expected": 4, "desc": "单源扩散：右下角海洋最远"},
        {"grid": [[1,1,1],[1,1,1]], "expected": -1, "desc": "全是陆地"},
        {"grid": [[0,0,0],[0,0,0]], "expected": -1, "desc": "全是海洋"},
        {"grid": [[1,0],[0,0]], "expected": 2, "desc": "2x2 扩散"},
        {"grid": [[1,0,0,0,1], [1,0,0,0,1], [1,0,0,0,1], [1,0,0,0,1], [1,0,0,0,1]], "expected": 2, "desc": "陆地在两端"},
        {"grid": [[0,0,1,0,0], [0,0,1,0,0], [0,0,1,0,0], [0,0,1,0,0], [0,0,1,0,0]], "expected": 2, "desc": "陆地在中间"},
        {"grid": [[1]], "expected": -1, "desc": "1x1 矩阵（无论 0 或 1 均不满足题目“陆地和海洋同时存在”）"},
        {"grid": [[1,0,0],[0,0,0],[0,0,1]], "expected": 2, "desc": "对角线分布"},
        {"grid": [[1,1,1],[1,0,1],[1,1,1]], "expected": 1, "desc": "密集陆地，少量海洋"}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 50)
    for i, tc in enumerate(test_cases):
        grid_input = [row[:] for row in tc["grid"]]
        actual = func(grid_input)
        status = "✅ PASS" if actual == tc["expected"] else "❌ FAIL"
        if actual == tc["expected"]: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")
    print("-" * 50)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1162(Solution1162().maxDistance)

ID   | 结果     | 描述
--------------------------------------------------
1    | ✅ PASS | 标准多源：中间海洋最远
2    | ✅ PASS | 单源扩散：右下角海洋最远
3    | ✅ PASS | 全是陆地
4    | ✅ PASS | 全是海洋
5    | ✅ PASS | 2x2 扩散
6    | ✅ PASS | 陆地在两端
7    | ✅ PASS | 陆地在中间
8    | ✅ PASS | 1x1 矩阵（无论 0 或 1 均不满足题目“陆地和海洋同时存在”）
9    | ✅ PASS | 对角线分布
10   | ✅ PASS | 密集陆地，少量海洋
--------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

1. **多源起点初始化**：
   - 题目求“海洋离最近陆地的距离的最大值”，本质是看**最后一批被淹没的海洋**在哪里。
   - 遍历矩阵，将所有 **陆地 (1)** 的坐标入队，并将它们的初始距离设为 0。
   - **特殊判定**：如果队列为空（全是海洋）或队列大小等于 $n^2$（全是陆地），说明没有可以计算的海洋，直接返回 -1。

2. **层序扩散逻辑**：
   - 从所有陆地同时出发。每扩散一层，距离 +1。
   - 使用 `max_dist` 记录过程中更新的最大距离。
   - 凡是 `dist[nr][nc] == -1` 的海洋，说明是第一次被触达，更新其距离并入队。

3. **结果判定**：
   - 最后一个出队的海洋所拥有的距离，即为全局“最远的最短距离”。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(n^2)$。每个格子仅入队出队一次。
* **空间复杂度**：$O(n^2)$。
:::